In [ ]:
# --- Imports ---

In [ ]:
import torch, torch.nn as nn, torch.optim as optim

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
from torchvision import transforms

In [ ]:
from src.dataset import PapyriDataset

In [ ]:
from src.model import create_resnet_regressor

In [ ]:
from src.utils import save_loss_plot

In [ ]:
import time, copy

In [ ]:
# --- Setup ---

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"Using device: {device}")

In [ ]:
# --- Hyperparameters ---

In [ ]:
BATCH_SIZE = 32

In [ ]:
LEARNING_RATE = 0.001

In [ ]:
NUM_EPOCHS = 25

In [ ]:
# --- Data Transforms ---

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# --- DataLoaders ---

In [ ]:
train_dataset = PapyriDataset(csv_file='data/train.csv', img_dir='data/images', transform=data_transforms['train'])

In [ ]:
val_dataset = PapyriDataset(csv_file='data/val.csv', img_dir='data/images', transform=data_transforms['val'])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

In [ ]:
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
# --- Model, Loss, Optimizer ---

In [ ]:
model = create_resnet_regressor().to(device)

In [ ]:
criterion = nn.MSELoss()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# --- Training Loop ---

In [ ]:
best_val_loss = float('inf')

In [ ]:
train_losses, val_losses = [], []

In [ ]:
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    # Training phase
    model.train()
    running_train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)

    # Validation phase
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * inputs.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Time: {(time.time() - start_time):.2f}s")

    # Save best model
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, 'checkpoints/best_model.pth')
        print(f"  -> New best model saved with Val Loss: {best_val_loss:.4f}")

print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

In [ ]:
# --- Save Loss Curve ---

In [ ]:
save_loss_plot(train_losses, val_losses)